# Satellite validation and calibration of the offshore hindcast

Compares the offshore hindcast at the goals (`offshore_waves`, e.g. WHACS) with
satellite altimetry, and calibrates it per goal:

0. **Partition energy** — the share of the bulk energy the hindcast partitions
   hold; the calibration closes them on the bulk (optional, `close_partitions`,
   default on; the steps below and the calibrated hindcast use the closed partitions).
1. **Passes at each goal** — altimeter observations within `radius_km` of each
   goal's hindcast node, in water deeper than `min_depth`, with good quality
   flags; one median Hs per satellite pass; the hindcast at the pass time.
2. **Calibration per goal** — directional calibration (wind sea and swell, per
   direction sector; Mínguez et al., 2011) fitted on `train_fraction` of the
   passes (75 % by default) and validated on the rest.
3. **Storm tail per goal** — one quantile map of the bulk Hs per goal, on its
   own passes; the bulk and every partition are scaled by the same ratio.
4. **Calibrated hindcast** — written as a new zarr, to add to `offshore_waves`
   (e.g. `WHACS_cal`) and use in 08 onwards like any other offshore source.
5. **Buoy check** — raw and calibrated Hs against deep buoys near the goals,
   which the calibration never saw.

Everything comes from the `satellite` block of the case JSON. Nothing here is
specific to one hindcast: any `offshore_waves` entry on `(time, goal)` with
`hs`, `dir` (and partitions `phs{i}`, `pdir{i}`) works.

**Prerequisites:** `07a_Offshore_WHACS.ipynb` or `07a_Offshore_ERA5.ipynb` (the goal
zarr) and the altimeter store (`satellite.data_path`). Method:
`bluemath_tk.waves.calibration` and `bluemath_tk.waves.altimetry`.

In [1]:
import json
import os
import sys

import pandas as pd

# GDAL/PROJ data paths when the kernel was not started through `conda activate`
os.environ.setdefault("GDAL_DATA", os.path.join(sys.prefix, "share", "gdal"))
os.environ.setdefault("PROJ_DATA", os.path.join(sys.prefix, "share", "proj"))

from utils import calibration as cal
from utils import viz
from utils.config import open_offshore_waves

## Configuration

In [2]:
offshore_name = "WHACS"  # offshore_waves entry to validate and calibrate (e.g. "ERA5")

cfg = cal.config()
print(json.dumps(cfg, indent=1))

{
 "name": "IMOS altimeter",
 "radius_km": 25,
 "min_depth": 15,
 "flags": [
  1,
  2
 ],
 "max_time_diff": "30min",
 "max_gap": "5min",
 "min_points": 3,
 "train_fraction": 0.75,
 "split": "random",
 "seed": 0,
 "sector_size": 22.5,
 "min_samples": 20,
 "tail_quantile": 0.999,
 "min_tail_samples": 5,
 "close_partitions": true,
 "data_path": "https://geoocean.unican.es/zarr/IMOS/imos_altimeter_dm00.zarr"
}


## 0. Partition energy

The spectral reconstruction sums the goal contributions of each partition, so the
partitions must hold the bulk energy. A hindcast may write fewer systems than its
spectrum has: WHACS (WAVEWATCH III default partitioning) writes the wind sea and
the three largest swells, so the energy of any further swell is in `hs` but in no
`phs{i}` (and WAVEWATCH III also drops partitions below a minimum Hs). The table
gives, per goal, the share of the bulk energy the partitions hold, over every hour
and split by whether the last partition slot holds energy: when the loss comes from
systems that are not written, it sits in the hours with the last slot used.

Unless `close_partitions` is `false` in the `satellite` block (default `true`), the
calibration closes them: every step below reads the hindcast with each hour's
partitions scaled by `sqrt(hs² / Σ phs²)` (`scale_partitions_to_bulk`; periods,
directions and spreads kept), which hands the missing energy to the systems that
are there, in proportion to theirs. The calibrated hindcast (`*_cal`) is therefore
closed; the source entry keeps the partitions as the hindcast gives them (a
spectral reconstruction from it starts low by this share). A hindcast without
partitions has nothing to close (the table is empty and the calibration works on
the bulk Hs and direction).

In [3]:
energy_share = cal.partition_energy_share(offshore_name)
energy_share.round(4)

,partitions,energy_share,hours_last_full,energy_share_last_full,energy_share_others
goal,,,,,
1,4,0.9995,0.1399,0.9928,0.9997
2,4,0.9996,0.1190,0.9948,0.9998
3,4,0.9997,0.1347,0.9949,0.9999
4,4,0.9996,0.1580,0.9938,0.9999
5,4,0.9994,0.2043,0.9933,0.9999
6,4,0.9993,0.2756,0.9946,0.9999
7,4,0.9993,0.2542,0.9940,0.9999
8,4,0.9995,0.2077,0.9938,0.9999
9,4,0.9996,0.1769,0.9946,0.9999


## 1. Satellite passes at each goal

Cached in `outputs/calibration/{offshore_name}/pairs.csv`; `refresh=True`
recomputes them (e.g. after changing `radius_km` or `min_depth`).

In [4]:
pairs = cal.satellite_pairs(offshore_name, refresh=False)
pairs.groupby("goal").size().rename("passes").to_frame().T

goal,1,10,2,3,4,5,6,7,8,9
passes,1729,1786,2514,1549,2240,4548,3446,1873,3456,3353


In [5]:
fig = viz.plot_satellite_pairs(pairs, cal.goal_nodes(offshore_name), cfg["radius_km"])

## 2. Calibration per goal

Fitted on `train_fraction` of each goal's passes; the metrics below are on the
**test** passes, which the fit never saw (bias = hindcast − satellite).

In [6]:
calibrations = cal.fit_calibration(offshore_name)
metrics = cal.calibration_metrics(offshore_name)
metrics[metrics["set"] == "test"].pivot(
    index="goal", columns="stage", values=["n", "bias", "rmse", "slope"]
).round(3)

n               bias              rmse             slope       
stage calibrated     raw calibrated    raw calibrated    raw calibrated    raw
goal                                                                          
1          432.0   432.0      0.002 -0.041      0.244  0.241      0.963  0.942
10         446.0   446.0      0.077  0.002      0.262  0.243      0.996  0.906
2          628.0   628.0     -0.018 -0.072      0.222  0.229      0.980  0.936
3          387.0   387.0      0.038 -0.042      0.243  0.243      0.922  0.873
4          559.0   559.0      0.052 -0.070      0.244  0.268      0.949  0.832
5         1136.0  1136.0      0.048 -0.107      0.258  0.300      0.930  0.814
6          859.0   859.0      0.062 -0.085      0.237  0.261      0.957  0.851
7          468.0   468.0      0.038 -0.060      0.266  0.279      0.938  0.853
8          864.0   864.0      0.029 -0.063      0.275  0.303      0.916  0.835
9          838.0   838.0      0.047 -0.048      0.247  0.261      0.980  0.875

In [7]:
fig = viz.plot_calibration_metrics(metrics, subset="test")

Per goal: test passes before and after the calibration, and the Hs factor of
each direction sector for wind sea and swell (red above 1, blue below, grey =
not enough data, left at 1).

In [8]:
for gid, calval in calibrations.items():
    calval.plot(title=f"{offshore_name} · goal {gid}")

## 3. Storm tail: quantile mapping per goal

A factor per sector fixes the level, not a tail that is too short: storms need
more than calm seas. One quantile map per goal, on that goal's passes, brings its
directionally calibrated Hs to the altimeter distribution, with a constant offset
beyond the goal's tail quantile. That quantile is the highest that keeps
`min_tail_samples` passes above it (default 5), at most `tail_quantile` (default
0.999): with one to a few thousand passes per goal it falls between about p99.4
and p99.9, so
each storm offset rests on a handful of passes. Neighbouring goals share passes
when their `radius_km` circles overlap.

The table gives, per goal, the passes, the tail quantile and the Hs of the
satellite and of each stage (`raw`, `hindcast` = after the directional step,
`mapped`) at a few probabilities; `n_above` is how many passes the value rests on.

**Fitted on all the passes** (train and test of section 2), so everything in
this section is in-sample: the deep buoys (section 5) and `10_Validate_Buoys`
are the independent check.

In [9]:
mappings = cal.fit_quantile_mapping(offshore_name)
qm_pairs = cal.mapped_pairs(offshore_name)
pd.concat(
    {
        gid: m.quantiles(raw=qm_pairs.loc[qm_pairs["goal"] == gid, "hs_raw"].to_numpy())
        .assign(n=len(m.evaluated_samples()), tail_quantile=m.tail_quantile)
        for gid, m in mappings.items()
    },
    names=["goal"],
).round(4)

reference     raw  hindcast  mapped  n_above     n  \
goal probability                                                       
1    0.500           0.9470  0.9380    0.9857  0.9476      864  1728   
     0.900           2.0082  1.9012    1.9634  2.0082      173  1728   
     0.990           3.3411  3.1375    3.2318  3.3320       17  1728   
     0.999           3.8985  3.5844    3.6144  3.7616        2  1728   
2    0.500           1.1480  1.1060    1.1555  1.1480     1256  2511   
     0.900           2.2695  2.2440    2.3421  2.2695      251  2511   
     0.990           3.7203  3.3784    3.5183  3.7200       25  2511   
     0.999           4.6296  4.2052    4.3757  4.4001        3  2511   
3    0.500           1.2040  1.1840    1.2430  1.2039      774  1548   
     0.900           2.4916  2.3666    2.5081  2.4912      155  1548   
     0.990           3.7242  3.3322    3.5443  3.7229       15  1548   
     0.999           5.0496  4.2450    4.5115  4.6656        2  1548   
4    0.500           1.2380  1.2440    1.3337  1.2380     1118  2237   
     0.900           2.6608  2.4568    2.7210  2.6606      224  2237   
     0.990           4.3299  3.4294    3.8728  4.3273       22  2237   
     0.999           5.3563  4.1144    4.6361  5.2646        2  2237   
5    0.500           1.3175  1.2880    1.4118  1.3176     2272  4544   
     0.900           2.7744  2.5880    2.9067  2.7746      454  4544   
     0.990           4.4790  3.6788    4.2243  4.4804       45  4544   
     0.999           5.9537  4.4257    5.1851  5.8962        5  4544   
6    0.500           1.3980  1.3550    1.4816  1.3980     1718  3436   
     0.900           3.0660  2.8380    3.1594  3.0647      344  3436   
     0.990           5.1248  4.2457    4.8642  5.1287       34  3436   
     0.999           7.1964  4.9996    5.9921  7.0311        3  3436   
7    0.500           1.3505  1.3760    1.4477  1.3505      936  1872   
     0.900           2.9150  2.7938    3.0155  2.9150      187  1872   
     0.990           4.4824  3.8924    4.4477  4.4831       19  1872   
     0.999           6.2966  4.5864    5.1140  5.7947        2  1872   
8    0.500           1.3160  1.2980    1.3704  1.3160     1727  3454   
     0.900           2.7840  2.6634    2.8619  2.7840      345  3454   
     0.990           4.5675  3.9326    4.3486  4.5673       35  3454   
     0.999           6.2276  4.5904    5.2219  6.0719        3  3454   
9    0.500           1.2620  1.2580    1.3327  1.2620     1676  3351   
     0.900           2.6150  2.5440    2.7510  2.6150      335  3351   
     0.990           4.5410  3.8200    4.3277  4.5410       34  3351   
     0.999           6.0949  4.5069    5.2215  6.0613        3  3351   
10   0.500           1.1630  1.2120    1.2261  1.1626      893  1786   
     0.900           2.4570  2.4590    2.6058  2.4565      179  1786   
     0.990           4.1510  3.7023    4.0298  4.1512       18  1786   
     0.999           6.5286  4.8198    5.5732  6.4263        2  1786   

                  tail_quantile  
goal probability                 
1    0.500               0.9971  
     0.900               0.9971  
     0.990               0.9971  
     0.999               0.9971  
2    0.500               0.9980  
     0.900               0.9980  
     0.990               0.9980  
     0.999               0.9980  
3    0.500               0.9968  
     0.900               0.9968  
     0.990               0.9968  
     0.999               0.9968  
4    0.500               0.9978  
     0.900               0.9978  
     0.990               0.9978  
     0.999               0.9978  
5    0.500               0.9989  
     0.900               0.9989  
     0.990               0.9989  
     0.999               0.9989  
6    0.500               0.9985  
     0.900               0.9985  
     0.990               0.9985  
     0.999               0.9985  
7    0.500               0.9973  
     0.900               0.9973  
     0.990               0.9973  
     0.

How to read the four panels (one figure per goal):

- **(a) the map**: the correction added to the directional Hs, as a function of
  it. Below the first fitted quantile (p1) it is a factor, between the fitted
  quantiles (dots) a linear interpolation, beyond the goal's tail quantile a constant
  offset: every storm above it gets the same extra metres, set by the few
  passes counted in the red band. The grey histogram shows how many passes
  support each part of the curve.
- **(b) Q-Q**: hindcast quantiles against satellite quantiles; on the 1:1 line
  the two distributions agree. The directional step moves the level, the map
  bends the tail.
- **(c) tail**: the same on an exceedance-probability (log) axis, where the
  storms are visible.
- **(d) pairs**: mapped Hs of each pass against its satellite Hs. The map
  matches distributions, not passes: compare the RMSE with the directional
  step in section 2.

In [10]:
for gid, m in mappings.items():
    m.plot(
        raw=qm_pairs.loc[qm_pairs["goal"] == gid, "hs_raw"].to_numpy(),
        labels={"reference": "satellite", "hindcast": "directional"},
        title=f"{offshore_name} · goal {gid} · quantile mapping (tail p{100 * m.tail_quantile:.2f})",
    )

**All goals side by side**: Hs quantile error (hindcast − satellite) per goal for
the three stages. `mapped` sits near zero at every goal by construction (each map
is fitted on its own passes); what to look at is how much each goal needed
(`raw` and `directional`) and how the corrections change from one goal to the
next. At p99 each goal has only ~10–40 passes above, so expect noise there.

In [11]:
fig = viz.plot_quantile_mapping_goals(qm_pairs)

## 4. Write the calibrated hindcast

Applies each goal's factors and its tail map to the whole record (partition Hs
times its sector factor, bulk Hs rescaled, then both by the tail ratio; periods
and directions unchanged) and writes
`*_cal.zarr` next to the source. Add the printed entry to `offshore_waves` in
the case JSON.

In [12]:
entry = cal.write_calibrated_hindcast(offshore_name)
print(json.dumps(entry, indent=4))

{
    "name": "WHACS_cal",
    "data_path": "inputs/WHACS/processed/Netherlands_WHACS_goal_cal.zarr",
    "hindcast": "WHACS"
}


**What the calibration does to the whole record**: exceedance curves of the full
hourly series at each goal, raw and calibrated, with the satellite passes. In
the grey band (rarer than one pass) no satellite checks the record: there the
calibrated tail is the shape of the hindcast tail plus the constant offset.

In [13]:
raw_hs = open_offshore_waves(offshore_name)["hs"].load()
calibrated_hs = open_offshore_waves(entry=entry)["hs"].load()
fig = viz.plot_calibrated_record(raw_hs, calibrated_hs, pairs)

## 5. Independent check: deep buoys near the goals

The satellite fitted both steps; the buoys did not. Buoys deeper than
`buoy_min_depth` and within `buoy_max_distance_km` of their closest goal node
measure nearly the same sea as the hindcast node, so raw and calibrated Hs there
(closest goal, no propagation) show whether the calibration helps. Metrics on
the hours both have (bias = hindcast − buoy); `q{p}` is the error at that Hs
quantile, the tail check (do not bin errors by observed Hs: regression to the
mean overstates the storm bias).

In [14]:
buoy_max_distance_km = 50  # to the closest goal node
buoy_min_depth = 24  # m, mesh depth at the buoy

buoy_sites = cal.buoy_check_sites(offshore_name, max_distance_km=buoy_max_distance_km, min_depth=buoy_min_depth)
buoy_sites.round(1)

,site,goal,distance_km,depth
0,buoy_EierlandseGat,5,27.1,27.9
1,buoy_EurogeulDWE,2,9.3,30.4
2,buoy_EurogeulE13,2,48.7,24.3
3,buoy_EurogeulE5,2,27.9,27.8
4,buoy_Europlatform,2,17.4,29.4
5,buoy_IJmuidenMunitiestort_1,3,38.1,24.7
6,buoy_L91,5,32.8,24.7
7,buoy_Q1,4,12.7,26.7
8,buoy_Schouwenbank,2,36.7,26.9


In [15]:
buoy_hs, buoy_estimates = cal.buoy_check(offshore_name, entry, buoy_sites)
buoy_metrics = cal.buoy_check_metrics(buoy_hs, buoy_estimates)
buoy_metrics.groupby("estimate")[["bias", "rmse", "r2", "q50", "q99", "q99.9"]].median().round(3)

,bias,rmse,r2,q50,q99,q99.9
estimate,,,,,,
WHACS,-0.031,0.251,0.909,0.020,-0.279,-0.614
WHACS_cal,0.037,0.242,0.914,0.052,0.127,-0.153


In [16]:
figs = viz.plot_buoy_check(buoy_hs, buoy_estimates, buoy_sites, buoy_metrics)

## Next

- Add the entry above to `offshore_waves` in the case JSON.
- `08_Dynamic_Cases_Generation.ipynb` with `offshore_name = "WHACS_cal"` (the
  dynamic runs must be forced with the calibrated hindcast too, so that 09
  compares like with like), then 09–12 with the same name.
- `10_Validate_Buoys.ipynb` with both names: the buoys are an independent check
  of the calibration.